# 05 · PVR evaluation on Google Colab (free T4)

Runs the last step of the project (validation → best λ, LRS3 test, VoxMM test) on Colab instead of Kaggle.
Inputs come from your committed Kaggle notebooks; results are saved to **Google Drive → `MyDrive/pvr_runs`**,
so a disconnect loses nothing: just run the cells again from the top and each step continues where it stopped.

**One-time setup**
1. *Runtime → Change runtime type → T4 GPU.*
2. Click the 🔑 **Secrets** icon (left bar) and add, with *Notebook access* switched on:
   * `HF_TOKEN`: your Hugging Face read token (Llama 3.2 access).
   * Kaggle key: either `KAGGLE_API_TOKEN` (new-style token from kaggle.com → Settings → API → *Generate New Token*)
     **or** `KAGGLE_USERNAME` + `KAGGLE_KEY` (from a legacy `kaggle.json`).
3. The visual model lives in your teammate's notebook `yuvaksc/visual-autoavsr`. If Cell 2 cannot download it, open that
   notebook on Kaggle → *Output* → download `runs/visual_lrs3_autoavsr/best.pt` and upload it to **`MyDrive/pvr_runs/best.pt`**.

**Every session:** run all cells from the top (*Runtime → Run all*). Finished steps are skipped automatically.

In [ ]:
# Cell 1: setup (every session, ~2 min)
import os, sys, glob, json, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/pvr_runs'          # results live here and survive disconnects
os.makedirs(RUNS, exist_ok=True)

for k in ['HF_TOKEN', 'KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        pass
assert os.environ.get('HF_TOKEN'), 'add HF_TOKEN in Secrets (key icon) and switch on Notebook access'
assert os.environ.get('KAGGLE_API_TOKEN') or os.environ.get('KAGGLE_KEY'), 'add your Kaggle key in Secrets'

if not os.path.exists('/content/repo'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/durgaanuhya02/PVR-Lip-Reading.git', '/content/repo'], check=True)
os.chdir('/content/repo')
!pip uninstall -y -q espnet
!pip install -q -U cmudict peft bitsandbytes kaggle
if not os.path.exists('/content/auto_avsr'):
    !git clone -q https://github.com/mpc001/auto_avsr /content/auto_avsr
    !cd /content/auto_avsr && git checkout -q 182b62837773ab01052d4ac21ef1d2203ea7d267
# the visual checkpoint's saved config points at /kaggle/working/auto_avsr
os.makedirs('/kaggle/working', exist_ok=True)
if not os.path.exists('/kaggle/working/auto_avsr'):
    os.symlink('/content/auto_avsr', '/kaggle/working/auto_avsr')

RESUME = '--resume' if '--resume' in open('src/pvr/run.py').read() else ''
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print('resume support:', bool(RESUME))
print('1 SETUP OK')

In [ ]:
# Cell 2: download the saved outputs of your Kaggle notebooks (every session, ~5-10 min)
DATA = '/content/data'
SOURCES = {                                   # Kaggle notebook -> what it holds
    'durgagurram/llm-train':       'LLM adapter',
    'durgagurram/lrs3-test':       'LRS3 test mouth clips',
    'durgagurram/voxmm-test-2':    'VoxMM test mouth clips',
    'durgagurram/lrs3-trainval-0': 'LRS3 train/val clips (validation)',
    'durgagurram/lrs3-trainval-1': 'LRS3 train/val clips (validation)',
    'durgagurram/lrs3-trainval-2': 'LRS3 train/val clips (validation)',
    'yuvaksc/visual-autoavsr':     'visual model best.pt',
}
for ref, what in SOURCES.items():
    dest = f"{DATA}/{ref.split('/')[1]}"
    if os.path.exists(dest) and os.listdir(dest):
        print('already here:', ref); continue
    if ref.endswith('visual-autoavsr') and os.path.exists(f'{RUNS}/best.pt'):
        print('using best.pt from Drive'); continue
    os.makedirs(dest, exist_ok=True)
    print(f'downloading {ref} ({what}) ...', flush=True)
    r = subprocess.run(['kaggle', 'kernels', 'output', ref, '-p', dest, '-q'], capture_output=True, text=True)
    if r.returncode != 0:
        print('   FAILED:', (r.stderr or r.stdout).strip()[-300:])
!du -sh {DATA}/* 2>/dev/null
print('2 DOWNLOAD DONE')

In [ ]:
# Cell 3: find inputs + checks
def find(pattern):
    return sorted(set(glob.glob(f'{DATA}/**/{pattern}', recursive=True)))
idx       = find('index.jsonl')
TRAIN     = [p for p in idx if 'lrs3_trainval_shard' in p]
LRS3_TEST = [p for p in idx if 'lrs3_test_rois' in p]
VOXMM     = [p for p in idx if 'voxmm' in p.lower()]
VISUAL    = [p for p in find('best.pt') if 'visual_lrs3_autoavsr' in p] or \
            ([f'{RUNS}/best.pt'] if os.path.exists(f'{RUNS}/best.pt') else [])
ADAPTER   = [os.path.dirname(p) for p in find('adapter_model.safetensors')]
for name, v in [('TRAIN', TRAIN), ('LRS3_TEST', LRS3_TEST), ('VOXMM', VOXMM), ('VISUAL', VISUAL), ('ADAPTER', ADAPTER)]:
    print(name, v)
assert len(TRAIN) == 3 and LRS3_TEST and VOXMM and VISUAL and ADAPTER, 'an input is missing (see the [] above)'
print('VoxMM clips:', sum(1 for p in VOXMM for _ in open(p)))

LLM = 'meta-llama/Llama-3.2-1B'
TRAIN_ARGS, VOXMM_ARGS = ' '.join(TRAIN), ' '.join(VOXMM)
COMMON = f'--visual {VISUAL[0]} --llm {LLM} --adapter {ADAPTER[0]} --n 5 --viseme-source phonemes'
sys.path.insert(0, '/content/auto_avsr')
from espnet.nets.pytorch_backend.e2e_asr_conformer import E2E
print('3 INPUTS OK')

In [ ]:
# Cell 4: speed test, 5 clips (~2 min). Optional; skip it on later sessions.
!time python -m src.pvr.run {COMMON} --index {LRS3_TEST[0]} --split test --limit 5 --out /content/speedtest

In [ ]:
# Cell 5: validation (300 clips) -> best lambda  (~15-30 min, skipped once done)
VAL = f'{RUNS}/pvr_val'
if not os.path.exists(f'{VAL}/metrics.json'):
    !python -m src.pvr.run {COMMON} --index {TRAIN_ARGS} --split val --limit 300 --sweep --out {VAL} {RESUME}
sweep = json.load(open(f'{VAL}/metrics.json'))['lambda_sweep']
LAM = float(min(sweep, key=sweep.get))
print('best lambda:', LAM, sweep)

In [ ]:
# Cell 6: LRS3 test (1,321 clips, ~1-2 h, continues after a disconnect)
OUT = f'{RUNS}/pvr_lrs3_test'
if not os.path.exists(f'{OUT}/metrics.json'):
    !python -m src.pvr.run {COMMON} --index {LRS3_TEST[0]} --split test --lam {LAM} --sweep --out {OUT} {RESUME}
print('LRS3 test done')

In [ ]:
# Cell 7: VoxMM test (2,125 clips, ~2-3.5 h, continues after a disconnect)
OUT = f'{RUNS}/pvr_voxmm_test'
if not os.path.exists(f'{OUT}/metrics.json'):
    !python -m src.pvr.run {COMMON} --index {VOXMM_ARGS} --split test --lam {LAM} --sweep --out {OUT} {RESUME}
print('VoxMM test done')

In [ ]:
# Cell 8: all results (also saved to Drive as summary.json)
summary = {'best_lambda': LAM}
for name in ['pvr_val', 'pvr_lrs3_test', 'pvr_voxmm_test']:
    m = f'{RUNS}/{name}/metrics.json'
    summary[name] = json.load(open(m)) if os.path.exists(m) else 'not done yet'
json.dump(summary, open(f'{RUNS}/summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))